# 全合约 close 重采样

在同一套分钟记录上建立时间、成交量或成交额采样点，各真实合约共用采样位置。若采样点位于某分钟的 $2/3$ 处，所有合约都用该分钟各自的复权 log open 与 log close 插值，得到新的 log close。`open` 只用于插值；输出不构造开盘价、最高价或最低价。

本方法提供固定区间采样，以及以每个分钟末端为终点、向前取 K 个点的采样。输入为 [b02 全合约复权分钟 demo](../b02_MainContinuousAdjustment/b02_MainContinuousAdjustment_demo.parquet)，`adjusted_log_*` 已经取过自然对数。

## 1. 间隔与共同采样位置

“对应 N 分钟”按实际记录计数：从所选区间的第一条分钟开始，每 N 行取最后一条 close，末尾不足 N 行也保留。午休、隔夜和缺失分钟不补行，跨交易日不重置。n 条分钟共有 $M=\lceil n/N\rceil$ 个基准 close。

以成交额为例，$m_i$ 为分钟 i 的成交额，$q_i=\sum_{j\leq i}m_j$。本演示将同一分钟各合约成交额相加；也可显式选择主力成交额。选定 N 后，令 $a=q_{n-1}/M$，在 $a,2a,\ldots,Ma$ 处采样，恰好得到 M 个点。成交量使用相同计算。N 由使用者选择；下方同时预览 10 和 15。

通过 `searchsorted` 找到每个目标 u 所在的分钟 i，再计算

$$f=(u-q_{i-1})/(q_i-q_{i-1}),\qquad x_c(u)=x^{open}_{c,i}+f(x^{close}_{c,i}-x^{open}_{c,i}).$$

其中 $q_{-1}=0$，$x$ 是复权 log 价格。假设本分钟成交量／额均匀累积、log 价格由 open 线性走到 close；分钟数据不能还原真实逐笔路径。所有合约使用相同 i 与 f，不能各自按成交量重新找点。价格域 close 为 $\exp(x_c(u))$。

目标恰好等于累计边界时取该分钟 close；生成目标时产生的浮点舍入误差会在机器精度范围内归回边界。零成交量／额不推进采样，因此尾部若全为零，最后事件采样点停在首次达到总量的分钟 close。目标为 0 时取区间第一分钟 open。内部插值缺任一端点则为空；缺合约行也为空，不前后填充。没有主力偏移的价格沿用上游空值。分数时间用源分钟的 `(bar_at − 1分钟, bar_at]` 定位，不跨休市时段连线。

In [1]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

CLOSE_RESAMPLING_VERSION = "0.1.0"


In [2]:
def prepare_sampling_panel(
    adjusted_minute_df: pd.DataFrame, *, activity_column: str = "money",
    activity_scope: str = "all", start_date=None, end_date=None,
) -> dict:
    """把单品种复权长表整理为共同分钟网格与价格矩阵。

    Args:
        adjusted_minute_df: b02 输出，含时间、合约、主力身份、数量和 adjusted_log_open/close。
        activity_column: 采样依据，money（元）或 volume（手）。
        activity_scope: all 为同分钟全合约合计；main 为当时主力合约原值。
        start_date: 起始交易日，包含；None 使用全部输入。
        end_date: 结束交易日，包含；None 使用全部输入。

    Returns:
        minutes_df、contract_codes、activity 及形状为 [分钟, 合约] 的
        log_open/log_close/present。缺合约位置为 NaN，present 区分缺行与缺价格。
    """
    if activity_column not in {"money", "volume"} or activity_scope not in {"all", "main"}:
        raise ValueError("activity_column 选 money/volume，activity_scope 选 all/main")
    required = ["bar_at", "trading_date", "exchange_code", "underlying_code", "contract_code",
                "main_contract_code", "is_main", activity_column, "adjusted_log_open", "adjusted_log_close"]
    minute_df = adjusted_minute_df.loc[:, required].copy()
    if start_date is not None:
        minute_df = minute_df.loc[minute_df.trading_date >= pd.Timestamp(start_date).date()]
    if end_date is not None:
        minute_df = minute_df.loc[minute_df.trading_date <= pd.Timestamp(end_date).date()]
    if minute_df.empty:
        raise ValueError("所选交易日没有分钟数据")
    if minute_df[["exchange_code", "underlying_code"]].drop_duplicates().shape[0] != 1:
        raise ValueError("一次采样只处理一个交易所的一个品种")
    keys = ["bar_at", "contract_code"]
    if minute_df[keys + ["trading_date", "main_contract_code", "is_main"]].isna().any().any():
        raise ValueError("时间、合约或主力身份缺失")
    if minute_df.duplicated(keys).any():
        raise ValueError("同一分钟出现重复合约行")
    if str(minute_df.bar_at.dt.tz) != "Asia/Shanghai":
        raise ValueError("bar_at 必须使用 Asia/Shanghai 时区")
    minute_identity = ["bar_at", "trading_date", "main_contract_code"]
    minutes_df = minute_df[minute_identity].drop_duplicates().sort_values("bar_at").reset_index(drop=True)
    if minutes_df.bar_at.duplicated().any():
        raise ValueError("同一分钟的交易日或主力身份不一致")
    if not (minute_df.is_main == (minute_df.contract_code == minute_df.main_contract_code)).all():
        raise ValueError("is_main 与主力合约身份不一致")
    quantity_df = minute_df if activity_scope == "all" else minute_df.loc[minute_df.is_main]
    quantities = quantity_df[activity_column].to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(quantities).all() or (quantities < 0).any():
        raise ValueError("采样所用成交量／额必须非负且完整；缺值不按零处理")
    activity = quantity_df.groupby("bar_at", sort=False)[activity_column].sum().reindex(minutes_df.bar_at).to_numpy(dtype=float)
    if not np.isfinite(activity).all():
        raise ValueError("存在缺失主力数量的分钟，不能建立主力采样时钟")
    contract_codes = np.sort(minute_df.contract_code.unique())
    minute_index = pd.Index(minutes_df.bar_at).get_indexer(minute_df.bar_at)
    contract_index = pd.Index(contract_codes).get_indexer(minute_df.contract_code)
    shape = (len(minutes_df), len(contract_codes))
    log_open = np.full(shape, np.nan)
    log_close = np.full(shape, np.nan)
    present = np.zeros(shape, dtype=bool)
    for source_column, price_matrix in (("adjusted_log_open", log_open), ("adjusted_log_close", log_close)):
        prices = minute_df[source_column].to_numpy(dtype=float, na_value=np.nan)
        if np.isinf(prices).any():
            raise ValueError(f"{source_column} 含无限值")
        price_matrix[minute_index, contract_index] = prices
    present[minute_index, contract_index] = True
    return {"minutes_df": minutes_df, "contract_codes": contract_codes, "activity": activity,
            "log_open": log_open, "log_close": log_close, "present": present,
            "activity_column": activity_column, "activity_scope": activity_scope,
            "exchange_code": minute_df.exchange_code.iloc[0], "underlying_code": minute_df.underlying_code.iloc[0]}


In [3]:
def make_sampling_clock(activity):
    """将逐分钟成交量或成交额转换为累计采样刻度。

    参数：activity，一维、非空、有限且非负的逐分钟数值。
    返回：float64 累计值；零量分钟保留。正数累积被浮点精度吞掉时报错。
    """
    activity_values = np.asarray(activity, dtype=np.float64)
    if activity_values.ndim != 1 or activity_values.size == 0:
        raise ValueError("activity 必须是一维非空数组。")
    if not np.isfinite(activity_values).all() or (activity_values < 0).any():
        raise ValueError("activity 不能有空值、无穷值或负数。")
    with np.errstate(over="ignore"):
        cumulative_activity = np.cumsum(activity_values, dtype=np.float64)
    if not np.isfinite(cumulative_activity).all():
        raise ValueError("activity 累积溢出，请先统一缩放数值。")
    clock_increments = np.diff(cumulative_activity, prepend=0.0)
    if ((activity_values > 0) & (clock_increments == 0)).any():
        raise ValueError("正 activity 在累积时丢失，当前数值精度不能定位采样点。")
    return cumulative_activity


def locate_sample_positions(q, targets):
    """反查采样点所在分钟及分钟内比例。

    参数：q 为 make_sampling_clock 的一维累计值；targets 为任意形状的目标刻度。
    返回：与 targets 同形的 (minute_indices, fractions)。目标 0 取首分钟 open；
    精确分钟末端取该分钟 close。超出历史、NaN 或无穷目标返回 (-1, NaN)。
    """
    cumulative_activity = np.asarray(q, dtype=np.float64)
    if (cumulative_activity.ndim != 1 or cumulative_activity.size == 0
            or not np.isfinite(cumulative_activity).all()
            or cumulative_activity[0] < 0
            or (np.diff(cumulative_activity) < 0).any()):
        raise ValueError("q 必须是一维非空、有限、非负且不下降的累计值。")
    return _locate_validated_positions(cumulative_activity, targets)


def _locate_validated_positions(cumulative_activity, targets, roundoff=None):
    """定位已校验刻度；roundoff 仅供内部生成目标吸附运算误差内的累计边界。"""
    target_values = np.asarray(targets, dtype=np.float64)
    if roundoff is not None:
        error_bounds = np.broadcast_to(np.asarray(roundoff, dtype=np.float64), target_values.shape)
        if not np.isfinite(error_bounds).all() or (error_bounds < 0).any():
            raise ValueError("采样目标的浮点误差界必须有限且非负。")
        boundary_indices = np.searchsorted(cumulative_activity, target_values, side="left")
        left_boundaries = np.where(boundary_indices == 0, 0.0,
                                  cumulative_activity[np.maximum(boundary_indices - 1, 0)])
        right_boundaries = cumulative_activity[np.minimum(boundary_indices, len(cumulative_activity) - 1)]
        near_left = np.abs(target_values - left_boundaries) <= error_bounds
        near_right = np.abs(target_values - right_boundaries) <= error_bounds
        if (near_left & near_right & (left_boundaries != right_boundaries)).any():
            raise ValueError("浮点误差区间覆盖两个累计边界，无法可靠定位采样点。")
        target_values = np.where(near_left, left_boundaries,
                                 np.where(near_right, right_boundaries, target_values))
    valid_targets = (np.isfinite(target_values) & (target_values >= 0)
                     & (target_values <= cumulative_activity[-1]))
    # 无效目标先置零，避免索引越界；有效的零目标单独定义为首分钟开端。
    safe_targets = np.where(valid_targets, target_values, 0.0)
    minute_indices = np.searchsorted(cumulative_activity, safe_targets, side="left")
    left_activity = np.where(minute_indices == 0, 0.0,
                            cumulative_activity[np.maximum(minute_indices - 1, 0)])
    minute_activity = cumulative_activity[minute_indices] - left_activity
    fractions = np.zeros_like(target_values, dtype=np.float64)
    # 用累计刻度之差作分母，精确边界可稳定得到 1，零量平台不会发生除零。
    np.divide(safe_targets - left_activity, minute_activity, out=fractions,
              where=valid_targets & (safe_targets > 0))
    return (np.where(valid_targets, minute_indices, -1),
            np.where(valid_targets, fractions, np.nan))


def interpolate_log_closes(log_open, log_close, minute_indices, fractions):
    """在各合约同一分钟、同一比例处计算采样 log close。

    参数：log_open、log_close 为形状一致的分钟数组，可为一维单序列或二维
    [分钟, 合约]；minute_indices、fractions 为同形位置数组，缺历史用 -1/NaN。
    返回：一维输入时与位置同形；二维输入时在末尾增加合约维。缺报价保持 NaN。
    """
    open_values = np.asarray(log_open, dtype=np.float64)
    close_values = np.asarray(log_close, dtype=np.float64)
    if (open_values.ndim not in (1, 2) or open_values.shape != close_values.shape
            or open_values.shape[0] == 0):
        raise ValueError("log_open/log_close 必须是同形、非空的一维或二维数组。")
    source_indices = np.asarray(minute_indices)
    fraction_values = np.asarray(fractions, dtype=np.float64)
    if source_indices.dtype.kind not in "iu" or source_indices.shape != fraction_values.shape:
        raise ValueError("minute_indices 须为整数数组，且与 fractions 同形。")
    if ((source_indices < -1) | (source_indices >= len(open_values))).any():
        raise ValueError("minute_indices 越界；仅 -1 表示缺少历史。")
    valid_positions = source_indices >= 0
    if (valid_positions & (~np.isfinite(fraction_values)
                           | (fraction_values < 0) | (fraction_values > 1))).any():
        raise ValueError("有效采样位置的 fractions 必须在 [0, 1] 内。")
    safe_indices = np.maximum(source_indices, 0)
    sampled_open = open_values[safe_indices]
    sampled_close = close_values[safe_indices]
    if np.isinf(sampled_open).any() or np.isinf(sampled_close).any():
        raise ValueError("采样所需 log 价格不能是无穷值；缺报价请用 NaN。")
    if open_values.ndim == 2:
        fraction_values = fraction_values[..., None]
        valid_positions = valid_positions[..., None]
    sampled_log_close = sampled_close - sampled_open
    sampled_log_close *= fraction_values
    sampled_log_close += sampled_open
    # 端点只需要对应端报价，不能让另一端的 NaN 污染有效 open/close。
    sampled_log_close = np.asarray(sampled_log_close)
    np.copyto(sampled_log_close, sampled_open, where=fraction_values == 0)
    np.copyto(sampled_log_close, sampled_close, where=fraction_values == 1)
    np.copyto(sampled_log_close, np.nan, where=~valid_positions)
    return sampled_log_close


def preview_sampling_intervals(activity, equivalent_minutes=(10, 15)):
    """预览目标分钟数对应的 a，以及单分钟可能被采样多次的情况。

    参数：activity 为逐分钟成交量/额；equivalent_minutes 为候选正整数分钟数。
    返回：(summary_df, exceed_details_df)。摘要包含样本数、a、超阈值分钟数、
    实际多次采样分钟数；明细列出 activity > a 的分钟及其实际采样次数。
    每 N 行保留一个 close，末尾不足 N 行也保留，因此 M=ceil(分钟数/N)。
    """
    cumulative_activity = make_sampling_clock(activity)
    activity_values = np.asarray(activity, dtype=np.float64)
    if cumulative_activity[-1] <= 0:
        raise ValueError("总 activity 为零，无法校准正采样间隔。")
    interval_values = np.asarray(equivalent_minutes)
    if (interval_values.ndim != 1 or interval_values.size == 0
            or interval_values.dtype.kind not in "iu" or (interval_values <= 0).any()):
        raise ValueError("equivalent_minutes 必须是一维非空正整数序列。")
    summaries, exceed_details = [], []
    minute_count = len(activity_values)
    for interval in interval_values:
        sample_count = (minute_count + int(interval) - 1) // int(interval)
        threshold = cumulative_activity[-1] / sample_count
        sample_numbers = np.arange(1, sample_count + 1, dtype=np.float64)
        targets = threshold * sample_numbers
        targets[-1] = cumulative_activity[-1]
        if threshold <= 0 or (np.diff(targets) <= 0).any():
            raise ValueError("采样间隔小于浮点可分辨范围，不能生成独立采样点。")
        # 误差只按本次除法、乘法的浮点间距估计，不使用宽泛的相对容差。
        roundoff = np.spacing(targets) + sample_numbers * np.spacing(threshold)
        roundoff[-1] = 0.0
        if (roundoff >= threshold / 2).any():
            raise ValueError("a 过小，相邻采样点的浮点误差区间发生重叠。")
        minute_indices, _ = _locate_validated_positions(cumulative_activity, targets, roundoff)
        minute_sample_counts = np.bincount(minute_indices, minlength=minute_count)
        exceed_mask = activity_values > threshold
        summaries.append({
            "equivalent_minutes": int(interval), "sample_count": sample_count, "a": threshold,
            "exceed_minute_count": int(exceed_mask.sum()),
            "exceed_minute_fraction": float(exceed_mask.mean()),
            "multiple_sample_minute_count": int((minute_sample_counts > 1).sum()),
            "max_samples_in_minute": int(minute_sample_counts.max()),
        })
        exceed_details.append(pd.DataFrame({
            "equivalent_minutes": int(interval), "minute_index": np.flatnonzero(exceed_mask),
            "activity": activity_values[exceed_mask], "a": threshold,
            "sample_count_in_minute": minute_sample_counts[exceed_mask],
        }))
    return pd.DataFrame(summaries), pd.concat(exceed_details, ignore_index=True)


def resample_interval_closes(activity, log_open, log_close, *, equivalent_minutes):
    """对已选定的一段分钟序列进行等 activity 采样，只返回 close。

    参数：activity 为逐分钟成交量/额；log_open/log_close 为同序的 log 价格，
    允许二维合约面板；equivalent_minutes 为参照的正整数分钟数。
    返回：字典，含 a、sample_count、minute_index、fraction、log_close。
    样本数与每 N 行取 close、保留尾段的时间采样一致。
    """
    if (isinstance(equivalent_minutes, (bool, np.bool_))
            or not isinstance(equivalent_minutes, (int, np.integer)) or equivalent_minutes <= 0):
        raise ValueError("equivalent_minutes 必须是正整数。")
    cumulative_activity = make_sampling_clock(activity)
    if cumulative_activity[-1] <= 0:
        raise ValueError("总 activity 为零，无法校准正采样间隔。")
    if np.ndim(log_open) == 0 or len(log_open) != len(cumulative_activity):
        raise ValueError("log 价格分钟数必须与 activity 一致。")
    sample_count = (len(cumulative_activity) + int(equivalent_minutes) - 1) // int(equivalent_minutes)
    threshold = cumulative_activity[-1] / sample_count
    sample_numbers = np.arange(1, sample_count + 1, dtype=np.float64)
    targets = threshold * sample_numbers
    targets[-1] = cumulative_activity[-1]
    if threshold <= 0 or (np.diff(targets) <= 0).any():
        raise ValueError("采样间隔小于浮点可分辨范围，不能生成独立采样点。")
    roundoff = np.spacing(targets) + sample_numbers * np.spacing(threshold)
    roundoff[-1] = 0.0
    if (roundoff >= threshold / 2).any():
        raise ValueError("a 过小，相邻采样点的浮点误差区间发生重叠。")
    minute_indices, fractions = _locate_validated_positions(cumulative_activity, targets, roundoff)
    sampled_log_close = interpolate_log_closes(log_open, log_close, minute_indices, fractions)
    return {"a": threshold, "sample_count": sample_count, "minute_index": minute_indices,
            "fraction": fractions, "log_close": sampled_log_close}


def iter_backward_closes(activity, log_open, log_close, *, a, lookback_count,
                         block_size=512, endpoint_indices=None):
    """从每个指定分钟末端向后取 K 个等 activity 的 close，分块返回。

    参数：activity 为逐分钟成交量/额；log_open/log_close 为同序 log 价格；
    a 为已确定的正间隔；lookback_count 为 K，取 lag=1..K，不含当前点；
    block_size 为每块末端数量；endpoint_indices 为需计算的分钟下标，None 表示全部。
    返回：字典迭代器，每块含 endpoint_index、minute_index、fraction、log_close。
    位置形状为 [本块末端数, K]；二维价格的输出再增加合约维。历史不足保持 NaN。
    """
    cumulative_activity = make_sampling_clock(activity)
    if np.ndim(a) != 0 or not np.isfinite(a) or a <= 0:
        raise ValueError("a 必须是有限正数。")
    for parameter_name, parameter_value in (("lookback_count", lookback_count), ("block_size", block_size)):
        if (isinstance(parameter_value, (bool, np.bool_))
                or not isinstance(parameter_value, (int, np.integer)) or parameter_value <= 0):
            raise ValueError(f"{parameter_name} 必须是正整数。")
    if np.ndim(log_open) == 0 or len(log_open) != len(cumulative_activity):
        raise ValueError("log 价格分钟数必须与 activity 一致。")
    # 价格仅转换一次，避免每个块重复把列表或 float32 面板转换成 float64。
    open_values = np.asarray(log_open, dtype=np.float64)
    close_values = np.asarray(log_close, dtype=np.float64)
    if open_values.ndim not in (1, 2) or open_values.shape != close_values.shape:
        raise ValueError("log_open/log_close 必须是同形的一维或二维数组。")
    endpoints = (np.arange(len(cumulative_activity), dtype=np.intp) if endpoint_indices is None
                 else np.asarray(endpoint_indices))
    if (endpoints.ndim != 1 or endpoints.dtype.kind not in "iu"
            or ((endpoints < 0) | (endpoints >= len(cumulative_activity))).any()):
        raise ValueError("endpoint_indices 必须是一维、未越界的整数下标。")
    with np.errstate(over="ignore"):
        lag_numbers = np.arange(1, int(lookback_count) + 1, dtype=np.float64)
        lag_activity = float(a) * lag_numbers
    if not np.isfinite(lag_activity).all() or (np.diff(lag_activity) <= 0).any():
        raise ValueError("回溯刻度溢出或间隔小于浮点可分辨范围。")
    for start in range(0, len(endpoints), int(block_size)):
        block_endpoints = endpoints[start:start + int(block_size)]
        endpoint_activity = cumulative_activity[block_endpoints, None]
        targets = endpoint_activity - lag_activity
        # 大累计值减去很小的 a 可能不改变数值，必须拒绝伪造的重复位置。
        if ((targets[:, 0] == endpoint_activity[:, 0]) & (endpoint_activity[:, 0] > 0)).any():
            raise ValueError("a 小于当前累计刻度的浮点分辨率。")
        if ((np.diff(targets, axis=1) >= 0) & (targets[:, 1:] >= 0)).any():
            raise ValueError("回溯采样点因浮点精度发生重合。")
        # 计入累计末端、a*k 乘法和相减的 ULP；仅内部生成目标可吸附边界。
        roundoff = (np.spacing(endpoint_activity) + np.spacing(lag_activity)
                    + lag_numbers * np.spacing(float(a)) + np.spacing(np.abs(targets)))
        if ((roundoff >= float(a) / 2) & (targets >= -roundoff)).any():
            raise ValueError("a 过小，相邻回溯点的浮点误差区间发生重叠。")
        minute_indices, fractions = _locate_validated_positions(cumulative_activity, targets, roundoff)
        sampled_log_close = interpolate_log_closes(open_values, close_values, minute_indices, fractions)
        yield {"endpoint_index": block_endpoints, "minute_index": minute_indices,
               "fraction": fractions, "log_close": sampled_log_close}


In [4]:
def resample_time_closes(log_close, *, equivalent_minutes: int) -> dict:
    """每 N 条实际分钟记录取最后一条 close，保留不足 N 条的尾组。

    Args:
        log_close: 一维分钟序列或二维 [分钟, 合约] log close 数组。
        equivalent_minutes: 正整数 N；跨休市和交易日继续计数。
    Returns:
        a（分钟数）、sample_count、minute_index、fraction（均为1）和 log_close。
    """
    prices = np.asarray(log_close, dtype=float)
    if prices.ndim not in (1, 2) or len(prices) == 0 or np.isinf(prices).any():
        raise ValueError("log_close 必须为非空一维或二维数组，缺值用 NaN")
    if isinstance(equivalent_minutes, (bool, np.bool_)) or not isinstance(equivalent_minutes, (int, np.integer)) or equivalent_minutes <= 0:
        raise ValueError("equivalent_minutes 必须为正整数")
    positions = np.minimum(np.arange(equivalent_minutes, len(prices) + equivalent_minutes, equivalent_minutes), len(prices)) - 1
    return {"a": float(equivalent_minutes), "sample_count": len(positions), "minute_index": positions,
            "fraction": np.ones(len(positions)), "log_close": prices[positions]}


## 2. 每个分钟末端向后采样

对第 e 个分钟末端，从累计值 $q_e$ 向前定位 $q_e-a,q_e-2a,\ldots,q_e-Ka$。K 是向前取点的数量，第一列最近，最后一列最远；不把当前 close 算进这 K 个点。当前 close 可直接从输入取得。早于输入起点的位置保留空值，不借用未来数据或补点。

`iter_backward_closes` 只计算一次累计序列，再按分钟末端分批生成目标，整批定位和插值。每批价格数组为 `[末端数, K, 合约数]`，使用者可逐批交给后续计算。`block_size` 控制内存；不要为方便而把所有批次转成列表。

a 由调用者传入，函数不在每个末端重新估计。固定区间演示可用该区间校准 a；若用于历史预测，应先用预测时已知的数据确定 a，再传入采样函数。本页的全区间复权 demo 和全区间校准仅演示方法。

In [5]:
SAMPLED_CLOSE_SCHEMA = pa.schema([
    pa.field("axis", pa.string(), False), pa.field("sample_id", pa.int64(), False),
    pa.field("equivalent_minutes", pa.int32(), False), pa.field("a", pa.float64(), False),
    pa.field("exchange_code", pa.string(), False), pa.field("underlying_code", pa.string(), False),
    pa.field("contract_code", pa.string(), False), pa.field("main_contract_code", pa.string(), False),
    pa.field("is_main", pa.bool_(), False), pa.field("trading_date", pa.date32(), False),
    pa.field("source_bar_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("fraction", pa.float64(), False),
    pa.field("sample_at", pa.timestamp("ns", tz="Asia/Shanghai"), False),
    pa.field("log_close", pa.float64()), pa.field("close", pa.float64()),
    pa.field("price_status", pa.string(), False),
], metadata={b"method_name": b"close_resampling", b"method_version": CLOSE_RESAMPLING_VERSION.encode(),
             b"primary_key": b"axis,sample_id,contract_code", b"timezone": b"Asia/Shanghai",
             b"price_rule": b"linear interpolation of adjusted log open/close; close=exp(log_close)"})


def sampled_closes_to_long(panel: dict, sampled: dict, *, axis: str, equivalent_minutes: int) -> pd.DataFrame:
    """将固定区间结果转为全合约长表，仅输出源分钟存在的合约行。

    Args:
        panel: prepare_sampling_panel 返回值。
        sampled: 固定区间结果，含 minute_index、fraction、log_close、a。
        axis: time、volume 或 money；用于区分采样轴。
        equivalent_minutes: 此组采样所对应的分钟数。

    Returns:
        每采样点每合约一行；价格为空的已有合约行保留并标记 missing_price。
    """
    source_index = sampled["minute_index"]
    if source_index.ndim != 1 or (source_index < 0).any():
        raise ValueError("长表转换只接受具有有效源分钟的固定区间结果")
    sample_rows, contract_columns = np.nonzero(panel["present"][source_index])
    sampled_minutes_df = panel["minutes_df"].iloc[source_index[sample_rows]].reset_index(drop=True)
    fractions = sampled["fraction"][sample_rows]
    log_closes = sampled["log_close"][sample_rows, contract_columns]
    source_times = pd.DatetimeIndex(sampled_minutes_df.bar_at)
    sample_times = source_times - pd.Timedelta(minutes=1) + pd.to_timedelta(np.rint(fractions * 60e9).astype(np.int64), unit="ns")
    contract_codes = panel["contract_codes"][contract_columns]
    return pd.DataFrame({
        "axis": axis, "sample_id": sample_rows + 1, "equivalent_minutes": equivalent_minutes, "a": sampled["a"],
        "exchange_code": panel["exchange_code"], "underlying_code": panel["underlying_code"],
        "contract_code": contract_codes, "main_contract_code": sampled_minutes_df.main_contract_code,
        "is_main": contract_codes == sampled_minutes_df.main_contract_code.to_numpy(),
        "trading_date": sampled_minutes_df.trading_date, "source_bar_at": source_times,
        "fraction": fractions, "sample_at": sample_times, "log_close": log_closes, "close": np.exp(log_closes),
        "price_status": np.where(np.isfinite(log_closes), "sampled", "missing_price"),
    })


In [6]:
def save_sampling_demo(sampled_close_df: pd.DataFrame, *, demo_path: str | Path, demo_identity: dict) -> dict:
    """保存并回读采样 demo；相同身份和内容保留原文件及生成时间。

    Args:
        sampled_close_df: sampled_closes_to_long 产生的一个或多个采样轴长表。
        demo_path: 本方法 *_demo.parquet 文件路径。
        demo_identity: 输入内容、代码、参数、范围及运行环境记录。

    Returns:
        安装后的 identity、preserved_existing 和 file_bytes。
    """
    from a01_Methods.b02_MainContinuousAdjustment.b02_MainContinuousAdjustment import arrow_content_sha256

    demo_path = Path(demo_path)
    sort_keys = [("axis", "ascending"), ("sample_id", "ascending"), ("contract_code", "ascending")]
    output_table = pa.Table.from_pandas(sampled_close_df, schema=SAMPLED_CLOSE_SCHEMA, preserve_index=False)
    output_table = output_table.replace_schema_metadata(SAMPLED_CLOSE_SCHEMA.metadata).sort_by(sort_keys).combine_chunks()
    for field in SAMPLED_CLOSE_SCHEMA:
        if not field.nullable and output_table[field.name].null_count:
            raise ValueError(f"输出 {field.name} 不可为空")
    if sampled_close_df.duplicated(["axis", "sample_id", "contract_code"]).any():
        raise ValueError("采样输出主键重复")
    identity = {**demo_identity, "output_rows": output_table.num_rows,
        "output_schema_sha256": hashlib.sha256(SAMPLED_CLOSE_SCHEMA.serialize().to_pybytes()).hexdigest(),
        "output_content_sha256": arrow_content_sha256(output_table, SAMPLED_CLOSE_SCHEMA, sort_keys),
        "output_sort_keys": sort_keys}
    identity = json.loads(json.dumps(identity, ensure_ascii=False, allow_nan=False))
    preserved = False
    if demo_path.exists():
        existing_table = pq.read_table(demo_path)
        existing_metadata = existing_table.schema.metadata or {}
        existing_identity = json.loads(existing_metadata[b"demo_identity"])
        existing_schema = existing_table.schema.with_metadata({k: v for k, v in existing_metadata.items() if k != b"demo_identity"})
        existing_digest = arrow_content_sha256(existing_table, existing_schema, sort_keys)
        if existing_digest != existing_identity["output_content_sha256"]:
            raise RuntimeError("已有采样 demo 内容摘要不符，请核查文件")
        preserved = (existing_schema.equals(SAMPLED_CLOSE_SCHEMA, check_metadata=True)
                     and {k: v for k, v in existing_identity.items() if k != "generated_at_utc"} == identity)
        del existing_table
    if preserved:
        identity = existing_identity
    else:
        identity["generated_at_utc"] = datetime.now(timezone.utc).isoformat()
        output_table = output_table.replace_schema_metadata({**SAMPLED_CLOSE_SCHEMA.metadata,
            b"demo_identity": json.dumps(identity, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()})
        staged_path = demo_path.with_name(demo_path.name + ".staging")
        if staged_path.exists():
            raise RuntimeError(f"存在待核查的 staging：{staged_path}")
        pq.write_table(output_table, staged_path, compression="zstd", compression_level=7)
        staged_table = pq.read_table(staged_path)
        if not staged_table.equals(output_table, check_metadata=True):
            raise RuntimeError("采样 staging 回读不一致")
        del staged_table
        os.replace(staged_path, demo_path)
    installed_table = pq.read_table(demo_path)
    installed_identity = json.loads(installed_table.schema.metadata[b"demo_identity"])
    if (installed_identity != identity or installed_table.num_rows != identity["output_rows"]
            or not installed_table.schema.equals(SAMPLED_CLOSE_SCHEMA, check_metadata=False)
            or arrow_content_sha256(installed_table, SAMPLED_CLOSE_SCHEMA, sort_keys) != identity["output_content_sha256"]):
        raise RuntimeError("采样 demo 安装后回读不一致")
    return {"identity": identity, "preserved_existing": preserved, "file_bytes": demo_path.stat().st_size}


## 3. 读取 RB 复权分钟

先读取 b02 已保存结果，核对物理字段、生成定义和实际内容摘要。这里只使用已生成的复权价格，不重新计算主力或复权，也不核对上游策略参数。本演示使用输入的完整交易日范围；修改 `start_date/end_date` 即可在给定区间重采样。

In [7]:
import sys
import platform
from time import perf_counter
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")
print(sys.executable)

from a01_Methods.b02_MainContinuousAdjustment.b02_MainContinuousAdjustment import arrow_content_sha256

upstream_path = candidate_root / "04_Research/a01_Methods/b02_MainContinuousAdjustment/b02_MainContinuousAdjustment_demo.parquet"
upstream_table = pq.read_table(upstream_path)
upstream_metadata = upstream_table.schema.metadata
upstream_identity = json.loads(upstream_metadata[b"demo_identity"])
upstream_schema = upstream_table.schema.with_metadata({k: v for k, v in upstream_metadata.items() if k != b"demo_identity"})
if arrow_content_sha256(upstream_table, upstream_schema, upstream_identity["output_sort_keys"]) != upstream_identity["output_content_sha256"]:
    raise RuntimeError("b02 demo 实际内容摘要不符，请先重算上游")
upstream_notebook = json.loads(upstream_path.with_name("b02_MainContinuousAdjustment.ipynb").read_text(encoding="utf-8"))
upstream_source = "\n\n".join("".join(c["source"]) for c in upstream_notebook["cells"]
    if c["cell_type"] == "code" and {"export", "demo-dependency"}.issubset(c.get("metadata", {}).get("tags", [])))
if hashlib.sha256(upstream_source.encode()).hexdigest() != upstream_identity["method_source_sha256"]:
    raise RuntimeError("b02 生成定义已改变，请先重算上游 demo")
read_columns = ["bar_at", "trading_date", "exchange_code", "underlying_code", "contract_code", "main_contract_code",
                "is_main", "volume", "money", "adjusted_log_open", "adjusted_log_close"]
adjusted_minute_df = upstream_table.select(read_columns).to_pandas()
del upstream_table

start_date = "2023-10-25"
end_date = "2025-10-20"
activity_scope = "all"
equivalent_minutes = 15
lookback_count = 32
block_size = 512

preparation_started = perf_counter()
sampling_panel = prepare_sampling_panel(adjusted_minute_df, activity_column="money", activity_scope=activity_scope,
                                        start_date=start_date, end_date=end_date)
preparation_seconds = perf_counter() - preparation_started
display(pd.DataFrame([{"交易日起点": start_date, "交易日终点": end_date,
    "分钟数": len(sampling_panel["minutes_df"]), "合约数": len(sampling_panel["contract_codes"]),
    "价格矩阵MiB": round((sampling_panel["log_open"].nbytes + sampling_panel["log_close"].nbytes)/2**20, 2)}]))
display(adjusted_minute_df.head(3))


E:\anaconda3\envs\latitude_env_v2\python.exe


,交易日起点,交易日终点,分钟数,合约数,价格矩阵MiB
0,2023-10-25,2025-10-20,164385,36,90.3


,bar_at,trading_date,exchange_code,underlying_code,contract_code,main_contract_code,is_main,volume,money,adjusted_log_open,adjusted_log_close
0,2023-10-24 21:01:00+08:00,2023-10-25,XSGE,RB,RB2311.XSGE,RB2401.XSGE,False,399.0,1.437020e+07,8.188689,8.188967
1,2023-10-24 21:01:00+08:00,2023-10-25,XSGE,RB,RB2312.XSGE,RB2401.XSGE,False,1083.0,3.922647e+07,8.194506,8.198089
2,2023-10-24 21:01:00+08:00,2023-10-25,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,57823.0,2.106265e+09,8.201112,8.198914


## 4. 选择分钟数前的预览

预览给出对应的 a、样本数量、单分钟成交量／额超过 a 的分钟数和占比，并给出实际采样两次以上的分钟数与最多次数。**超过 a 的分钟数是风险提示，不等于实际多次采样的分钟数**，后者还取决于累计采样点的位置。

`exceed_details_df` 保留全部超过 a 的分钟，展示时只列前 10 条；每条含时间与实际采样次数。修改 `equivalent_minutes` 后重新运行采样即可。

In [8]:
minute_index = pd.Index(sampling_panel["minutes_df"].bar_at)
scoped_quantity_df = adjusted_minute_df.loc[adjusted_minute_df.bar_at.isin(minute_index)]
if activity_scope == "main":
    scoped_quantity_df = scoped_quantity_df.loc[scoped_quantity_df.is_main]
volume_values = scoped_quantity_df.volume.to_numpy(dtype=float, na_value=np.nan)
if not np.isfinite(volume_values).all() or (volume_values < 0).any():
    raise ValueError("成交量存在缺失或非法值，不能按零量参与汇总")
volume_activity = scoped_quantity_df.groupby("bar_at").volume.sum().reindex(minute_index).to_numpy(dtype=float)
activity_by_axis = {"money": sampling_panel["activity"], "volume": volume_activity}
preview_frames, exceed_frames = [], []
for axis, activity in activity_by_axis.items():
    preview_df, exceed_df = preview_sampling_intervals(activity, equivalent_minutes=sorted({10, 15, equivalent_minutes}))
    preview_df.insert(0, "axis", axis)
    exceed_df.insert(0, "axis", axis)
    exceed_df["bar_at"] = minute_index.take(exceed_df.minute_index.to_numpy(dtype=int))
    preview_frames.append(preview_df)
    exceed_frames.append(exceed_df)
preview_df = pd.concat(preview_frames, ignore_index=True)
exceed_details_df = pd.concat(exceed_frames, ignore_index=True)
display(preview_df)
display(exceed_details_df.head(10))


,axis,equivalent_minutes,sample_count,a,exceed_minute_count,exceed_minute_fraction,multiple_sample_minute_count,max_samples_in_minute
0,money,10,16439,1.943946e+09,383,0.002330,126,4
1,money,15,10959,2.916008e+09,107,0.000651,38,3
2,volume,10,16439,5.728073e+04,386,0.002348,142,4
3,volume,15,10959,8.592371e+04,106,0.000645,30,3


,axis,equivalent_minutes,minute_index,activity,a,sample_count_in_minute,bar_at
0,money,10,0,2.337158e+09,1.943946e+09,1,2023-10-24 21:01:00+08:00
1,money,10,120,2.721609e+09,1.943946e+09,1,2023-10-25 09:01:00+08:00
2,money,10,690,2.112648e+09,1.943946e+09,1,2023-10-26 21:01:00+08:00
3,money,10,1035,1.971545e+09,1.943946e+09,1,2023-10-27 21:01:00+08:00
4,money,10,1380,2.113083e+09,1.943946e+09,1,2023-10-30 21:01:00+08:00
5,money,10,2415,2.599521e+09,1.943946e+09,1,2023-11-02 21:01:00+08:00
6,money,10,4605,2.188924e+09,1.943946e+09,1,2023-11-13 09:01:00+08:00
7,money,10,4830,2.251905e+09,1.943946e+09,1,2023-11-13 21:01:00+08:00
8,money,10,5175,3.545353e+09,1.943946e+09,2,2023-11-14 21:01:00+08:00
9,money,10,5465,2.039542e+09,1.943946e+09,1,2023-11-15 14:06:00+08:00


## 5. 固定区间采样

下面分别按实际分钟数、成交量和成交额采样，三者采样点数相同。成交量／额的采样点可以在一分钟内出现多次；同一点的各合约使用相同源分钟和比例。

矩阵用 NaN 保留缺合约位置。保存为长表时，只保存源分钟实际有记录的合约；该行价格不足以插值时仍保留，`price_status=missing_price`。`source_bar_at` 与 `fraction` 共同表示采样位置，`sample_at` 是四舍五入到纳秒的便于阅读的时间。主力身份按源分钟记录，`is_main` 可直接选取主力 close。

In [9]:
fixed_started = perf_counter()
sampled_by_axis = {}
for axis, activity in activity_by_axis.items():
    sampled_by_axis[axis] = resample_interval_closes(activity, sampling_panel["log_open"], sampling_panel["log_close"],
                                                     equivalent_minutes=equivalent_minutes)
minute_count = len(sampling_panel["minutes_df"])
sampled_by_axis["time"] = resample_time_closes(sampling_panel["log_close"], equivalent_minutes=equivalent_minutes)
fixed_seconds = perf_counter() - fixed_started
sampled_close_df = pd.concat([sampled_closes_to_long(sampling_panel, sampled, axis=axis, equivalent_minutes=equivalent_minutes)
    for axis, sampled in sampled_by_axis.items()], ignore_index=True)
assert sampled_close_df.groupby("axis").sample_id.nunique().nunique() == 1
display(sampled_close_df.groupby("axis").agg(采样点数=("sample_id", "nunique"), 合约行数=("contract_code", "size"),
    缺价格行数=("log_close", lambda values: values.isna().sum())))
display(sampled_close_df.loc[sampled_close_df.is_main].groupby("axis").head(3))
# 展示同一分数分钟的多个合约；每个合约使用自己的 open/close，比例完全相同。
fractional_rows = sampled_close_df.loc[(sampled_close_df.axis == "money") & (sampled_close_df.fraction < 1)]
if fractional_rows.empty:
    print("此配置的成交额采样点均落在分钟末端。")
else:
    display(fractional_rows.loc[fractional_rows.sample_id == fractional_rows.sample_id.iloc[0]])


,采样点数,合约行数,缺价格行数
axis,,,
money,10959,131508,0
time,10959,131508,0
volume,10959,131508,0


,axis,sample_id,equivalent_minutes,a,exchange_code,underlying_code,contract_code,main_contract_code,is_main,trading_date,source_bar_at,fraction,sample_at,log_close,close,price_status
2,money,1,15,2.916008e+09,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.199434,3638.888409,sampled
14,money,2,15,2.916008e+09,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:09:00+08:00,0.606206,2023-10-24 21:08:36.372333761+08:00,8.198644,3636.016024,sampled
26,money,3,15,2.916008e+09,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:26:00+08:00,0.158482,2023-10-24 21:25:09.508910444+08:00,8.198972,3637.208556,sampled
131510,volume,1,15,8.592371e+04,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:03:00+08:00,0.240002,2023-10-24 21:02:14.400124702+08:00,8.199663,3639.723791,sampled
131522,volume,2,15,8.592371e+04,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:11:00+08:00,0.374584,2023-10-24 21:10:22.475021043+08:00,8.199092,3637.645424,sampled
131534,volume,3,15,8.592371e+04,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:30:00+08:00,0.627084,2023-10-24 21:29:37.625061184+08:00,8.198930,3637.058105,sampled
263018,time,1,15,1.500000e+01,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:15:00+08:00,1.000000,2023-10-24 21:15:00+08:00,8.199197,3638.028058,sampled
263030,time,2,15,1.500000e+01,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:30:00+08:00,1.000000,2023-10-24 21:30:00+08:00,8.198930,3637.058105,sampled
263042,time,3,15,1.500000e+01,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:45:00+08:00,1.000000,2023-10-24 21:45:00+08:00,8.199763,3640.088232,sampled


,axis,sample_id,equivalent_minutes,a,exchange_code,underlying_code,contract_code,main_contract_code,is_main,trading_date,source_bar_at,fraction,sample_at,log_close,close,price_status
0,money,1,15,2.916008e+09,XSGE,RB,RB2311.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.189214,3601.888389,sampled
1,money,1,15,2.916008e+09,XSGE,RB,RB2312.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.196744,3629.115567,sampled
2,money,1,15,2.916008e+09,XSGE,RB,RB2401.XSGE,RB2401.XSGE,True,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.199434,3638.888409,sampled
3,money,1,15,2.916008e+09,XSGE,RB,RB2402.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.198365,3635.002002,sampled
4,money,1,15,2.916008e+09,XSGE,RB,RB2403.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.203547,3653.888418,sampled
5,money,1,15,2.916008e+09,XSGE,RB,RB2404.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.200838,3644.002007,sampled
6,money,1,15,2.916008e+09,XSGE,RB,RB2405.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.203000,3651.888416,sampled
7,money,1,15,2.916008e+09,XSGE,RB,RB2406.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.196162,3627.001998,sampled
8,money,1,15,2.916008e+09,XSGE,RB,RB2407.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.203578,3654.002013,sampled
9,money,1,15,2.916008e+09,XSGE,RB,RB2408.XSGE,RB2401.XSGE,False,2023-10-25,2023-10-24 21:02:00+08:00,0.886418,2023-10-24 21:01:53.185103052+08:00,8.189523,3603.001985,sampled


## 6. 每个分钟向后取点

本例将成交额 a 固定为上一节的值，遍历全部分钟末端，每次向前取 32 点。只保留最后一个批次用于展示，完整计算结果可在循环内交给下游。价格列顺序见 `sampling_panel["contract_codes"]`。

对单条分钟序列，同一函数也接受一维 log open/close，输出 `[末端数, K]`；全合约输入则输出 `[末端数, K, 合约数]`。

In [10]:
backward_started = perf_counter()
endpoint_count = 0
missing_history_count = 0
max_batch_price_bytes = 0
for sampled_batch in iter_backward_closes(sampling_panel["activity"], sampling_panel["log_open"], sampling_panel["log_close"],
    a=sampled_by_axis["money"]["a"], lookback_count=lookback_count, block_size=block_size):
    endpoint_count += len(sampled_batch["endpoint_index"])
    missing_history_count += int((sampled_batch["minute_index"] < 0).sum())
    max_batch_price_bytes = max(max_batch_price_bytes, sampled_batch["log_close"].nbytes)
backward_seconds = perf_counter() - backward_started
assert endpoint_count == minute_count
last_positions = sampled_batch["minute_index"][-1]
last_source_times = pd.Series(pd.NaT, index=range(lookback_count), dtype="datetime64[ns, Asia/Shanghai]")
valid_history = last_positions >= 0
last_source_times.loc[valid_history] = minute_index.take(last_positions[valid_history]).as_unit("ns")
display(pd.DataFrame({"向前第几个点": np.arange(1, lookback_count + 1),
    "源分钟": last_source_times, "分钟内比例": sampled_batch["fraction"][-1]}).head(8))
display(pd.DataFrame([{"全部末端数": endpoint_count, "每末端向前点数": lookback_count,
    "历史不足位置数": missing_history_count, "最大单批价格MiB": round(max_batch_price_bytes / 2**20, 2)}]))


,向前第几个点,源分钟,分钟内比例
0,1,2025-10-20 14:24:00+08:00,0.046294
1,2,2025-10-20 13:47:00+08:00,0.945673
2,3,2025-10-20 11:17:00+08:00,0.157294
3,4,2025-10-20 10:42:00+08:00,0.420999
4,5,2025-10-20 10:02:00+08:00,0.493086
5,6,2025-10-20 09:28:00+08:00,0.567353
6,7,2025-10-20 09:04:00+08:00,0.640739
7,8,2025-10-17 22:51:00+08:00,0.524164


,全部末端数,每末端向前点数,历史不足位置数,最大单批价格MiB
0,164385,32,7178,4.5


## 7. 边界核对与计时

验证一分钟内多次采样、不同合约同位置插值、尾部计数、零成交量／额、恰好落在边界、缺价格、缺历史和不同批大小一致性。速度比较使用相同输入和输出，与独立逐点参考实现核对；耗时为本机本次运行结果。

In [11]:
from bisect import bisect_left

# 在一根分钟中采样多次；第二个合约与第一个合约的 log 价差始终为 1。
example_activity = np.array([0., 12., 0., 0., 0.])
example_open = np.column_stack([np.arange(5.), np.arange(5.) + 1])
example_close = example_open + 3
example_sampled = resample_interval_closes(example_activity, example_open, example_close, equivalent_minutes=1)
assert example_sampled["sample_count"] == 5
np.testing.assert_array_equal(example_sampled["minute_index"], np.ones(5, dtype=int))
np.testing.assert_allclose(example_sampled["fraction"], [.2, .4, .6, .8, 1.])
np.testing.assert_allclose(np.diff(example_sampled["log_close"], axis=1), 1)

# 2/3 分钟的所有合约都插值；例如 (1→4) 得3，(2→5) 得4。
example_indices, example_fractions = locate_sample_positions(make_sampling_clock(example_activity), [8.])
np.testing.assert_allclose(example_fractions, 2 / 3)
np.testing.assert_allclose(interpolate_log_closes(example_open, example_close, example_indices, example_fractions), [[3., 4.]])

boundary_indices, boundary_fractions = locate_sample_positions(make_sampling_clock([0., 2., 0., 3., 0.]), [0., 2., 3.5, 5., -1., 6.])
np.testing.assert_array_equal(boundary_indices, [0, 1, 3, 3, -1, -1])
np.testing.assert_allclose(boundary_fractions, [0., 1., .5, 1., np.nan, np.nan], equal_nan=True)
endpoint_prices = interpolate_log_closes([np.nan, 7.], [5., np.nan], np.array([0, 1, 0]), np.array([1., 0., .5]))
np.testing.assert_allclose(endpoint_prices, [5., 7., np.nan], equal_nan=True)
tail_example = resample_interval_closes(np.ones(31), np.zeros(31), np.ones(31), equivalent_minutes=15)
assert tail_example["sample_count"] == 3
np.testing.assert_array_equal(resample_time_closes(np.arange(31.), equivalent_minutes=15)["log_close"], [14., 29., 30.])
single_example = resample_interval_closes([4.], [1.], [2.], equivalent_minutes=15)
np.testing.assert_array_equal(single_example["log_close"], [2.])

for bad_activity in ([0., 0.], [1., -1.], [1., np.nan], [1e20, 1.]):
    try:
        resample_interval_closes(bad_activity, np.zeros(2), np.ones(2), equivalent_minutes=1)
    except ValueError:
        pass
    else:
        raise AssertionError("异常 activity 未被拒绝")

# 删除非主力的一条输入，验证不齐的合约面板不会前填或后填。
small_times = minute_index[:3]
small_panel_df = adjusted_minute_df.loc[adjusted_minute_df.bar_at.isin(small_times)].copy()
missing_row = small_panel_df.loc[(small_panel_df.bar_at == small_times[1]) & ~small_panel_df.is_main].index[0]
missing_contract = small_panel_df.loc[missing_row, "contract_code"]
small_panel_df = small_panel_df.drop(index=missing_row)
incomplete_panel = prepare_sampling_panel(small_panel_df)
missing_column = np.flatnonzero(incomplete_panel["contract_codes"] == missing_contract)[0]
assert not incomplete_panel["present"][1, missing_column]
assert np.isnan(interpolate_log_closes(incomplete_panel["log_open"], incomplete_panel["log_close"],
    np.array([1]), np.array([2 / 3]))[0, missing_column])
np.testing.assert_allclose(incomplete_panel["activity"], small_panel_df.groupby("bar_at").money.sum().to_numpy())


def scalar_backward_reference(activity, log_open, log_close, endpoints, a, count):
    """逐点参考：逐个末端与滞后定位，独立核对向量化结果。"""
    cumulative = np.cumsum(activity)
    prices = np.full((len(endpoints), count, log_open.shape[1]), np.nan)
    for endpoint_row, endpoint in enumerate(endpoints):
        for lag in range(1, count + 1):
            target = cumulative[endpoint] - a * lag
            if target < 0:
                continue
            index = bisect_left(cumulative, target)
            left = cumulative[index - 1] if index else 0.
            fraction = (target - left) / (cumulative[index] - left) if target > 0 else 0.
            if fraction == 0:
                prices[endpoint_row, lag - 1] = log_open[index]
            elif fraction == 1:
                prices[endpoint_row, lag - 1] = log_close[index]
            else:
                prices[endpoint_row, lag - 1] = log_open[index] + fraction * (log_close[index] - log_open[index])
    return prices


rng = np.random.default_rng(20261005)
random_activity = rng.integers(0, 10, size=101).astype(float)
random_open = rng.normal(8., .1, (101, 3))
random_close = random_open + rng.normal(0., .01, (101, 3))
random_close[3, 1] = np.nan
random_endpoints = np.arange(101)
reference_prices = scalar_backward_reference(random_activity, random_open, random_close, random_endpoints, 7., 9)
for test_block_size in (1, 17, 200):
    actual_prices = np.concatenate([batch["log_close"] for batch in iter_backward_closes(
        random_activity, random_open, random_close, a=7., lookback_count=9, block_size=test_block_size)])
    np.testing.assert_allclose(actual_prices, reference_prices, rtol=0, atol=2e-15, equal_nan=True)
# 追加未来数据不会影响固定 a 下既有末端的结果。
prefix_batch = next(iter_backward_closes(random_activity[:50], random_open[:50], random_close[:50],
    a=7., lookback_count=9, endpoint_indices=[49]))
np.testing.assert_allclose(prefix_batch["log_close"][0], reference_prices[49], rtol=0, atol=2e-15, equal_nan=True)

# 十进制边界不能因二进制浮点舍入而跨到下一分钟 open，或被误判为缺历史。
decimal_boundary = next(iter_backward_closes([.1, .2], [10., 30.], [20., 40.], a=.2, lookback_count=1, endpoint_indices=[1]))
np.testing.assert_allclose(decimal_boundary["log_close"], [[20.]])
decimal_origin = next(iter_backward_closes([.3], [10.], [20.], a=.1, lookback_count=3))
np.testing.assert_allclose(decimal_origin["log_close"][0, -1], 10.)

benchmark_endpoints = np.arange(max(0, minute_count - 1024), minute_count)
benchmark_started = perf_counter()
benchmark_reference = scalar_backward_reference(sampling_panel["activity"], sampling_panel["log_open"], sampling_panel["log_close"],
    benchmark_endpoints, sampled_by_axis["money"]["a"], 16)
reference_seconds = perf_counter() - benchmark_started
benchmark_started = perf_counter()
benchmark_vectorized = np.concatenate([batch["log_close"] for batch in iter_backward_closes(
    sampling_panel["activity"], sampling_panel["log_open"], sampling_panel["log_close"],
    a=sampled_by_axis["money"]["a"], lookback_count=16, endpoint_indices=benchmark_endpoints)])
vectorized_seconds = perf_counter() - benchmark_started
np.testing.assert_allclose(benchmark_vectorized, benchmark_reference, rtol=0, atol=2e-15, equal_nan=True)
assert not any(column in sampled_close_df.columns for column in ("open", "high", "low"))
assert not sampled_close_df.duplicated(["axis", "sample_id", "contract_code"]).any()
assert sampled_close_df.groupby(["axis", "sample_id"])[["source_bar_at", "fraction"]].nunique().eq(1).all().all()
print("边界、全合约共同位置、随机参考、分批一致性核对通过。")
display(pd.DataFrame([
    {"计算": f"输入整理（{minute_count}分钟 × {len(sampling_panel['contract_codes'])}合约）", "秒": preparation_seconds},
    {"计算": "全区间三种轴，位置与close数组", "秒": fixed_seconds},
    {"计算": f"全部分钟末端 × {lookback_count}点 × {len(sampling_panel['contract_codes'])}合约，分批消费", "秒": backward_seconds},
    {"计算": f"{len(benchmark_endpoints)}末端 × 16点 × {len(sampling_panel['contract_codes'])}合约，逐点参考", "秒": reference_seconds},
    {"计算": "相同输入输出，向量化", "秒": vectorized_seconds},
]))
print("本次内核提速倍数：", round(reference_seconds / vectorized_seconds, 2), "（不含读取和长表转换）")


边界、全合约共同位置、随机参考、分批一致性核对通过。


,计算,秒
0,输入整理（164385分钟 × 36合约）,2.331533
1,全区间三种轴，位置与close数组,0.025240
2,全部分钟末端 × 32点 × 36合约，分批消费,1.554877
3,1024末端 × 16点 × 36合约，逐点参考,0.046296
4,相同输入输出，向量化,0.007401


本次内核提速倍数： 6.26 （不含读取和长表转换）


## 8. 保存、复读与导入

保存三种采样轴的固定区间全合约 close 长表到同目录 demo。metadata 记录输入内容和上游身份、范围、采样规则、代码和运行环境；相同身份与内容重跑保留原文件。滚动大数组逐批消费，不另存一个全量副本。

导入时将项目根与 `04_Research` 加入 `sys.path`，再从 `a01_Methods.b03_CloseResampling.b03_CloseResampling` 导入所需函数。同名 `.py` 只加载本页的 `export` 定义，不读数据、不执行 demo。

In [12]:
method_notebook_path = candidate_root / "04_Research/a01_Methods/b03_CloseResampling/b03_CloseResampling.ipynb"
method_notebook = json.loads(method_notebook_path.read_text(encoding="utf-8"))
method_source = "\n\n".join("".join(c["source"]) for c in method_notebook["cells"]
    if c["cell_type"] == "code" and "demo-dependency" in c.get("metadata", {}).get("tags", []))
dependency_paths = ["04_Research/a00_notebook_loader.py", "config/data_contracts.py"]
sampling_identity = {
    "artifact_role": "method_demo", "producer_notebook": method_notebook_path.relative_to(candidate_root).as_posix(),
    "method_version": CLOSE_RESAMPLING_VERSION, "method_source_tag": "demo-dependency",
    "method_source_sha256": hashlib.sha256(method_source.encode()).hexdigest(),
    "dependency_source_sha256": {path: hashlib.sha256((candidate_root / path).read_bytes()).hexdigest() for path in dependency_paths},
    "upstream_demo_identity": upstream_identity, "upstream_path": upstream_path.relative_to(candidate_root).as_posix(),
    "scope": {"start_date": start_date, "end_date": end_date, "interval_closed": "both", "timezone": "Asia/Shanghai"},
    "parameters": {"equivalent_minutes": equivalent_minutes, "activity_scope": activity_scope,
        "axes": ["money", "volume", "time"], "a": {axis: sampled["a"] for axis, sampled in sampled_by_axis.items()},
        "sample_count": "ceil(observed_minutes/N); keep short final time group",
        "event_calibration": "a=total_activity/sample_count; all event intervals equal",
        "interpolation": "shared minute/fraction; linear adjusted log open to close",
        "boundary": "first cumulative boundary (side=left); origin=first open",
        "missing": "no fill; absent source contract rows omitted in long output; missing price rows retained"},
    "runtime": {"python": sys.version.split()[0], "python_executable": sys.executable,
        "numpy": np.__version__, "pandas": pd.__version__, "pyarrow": pa.__version__, "platform": platform.platform()},
}
sampling_demo_path = method_notebook_path.with_name(method_notebook_path.stem + "_demo.parquet")
save_summary = save_sampling_demo(sampled_close_df, demo_path=sampling_demo_path, demo_identity=sampling_identity)
print("保存并复读通过：", sampling_demo_path)
print("合约行数：", save_summary["identity"]["output_rows"], "保留已有文件：", save_summary["preserved_existing"])
display(pd.DataFrame([{"字段": f.name, "类型": str(f.type), "允许空值": f.nullable} for f in SAMPLED_CLOSE_SCHEMA]))


保存并复读通过： E:\Latitude_Analytics_v2\04_Research\a01_Methods\b03_CloseResampling\b03_CloseResampling_demo.parquet
合约行数： 394524 保留已有文件： True


,字段,类型,允许空值
0,axis,string,False
1,sample_id,int64,False
2,equivalent_minutes,int32,False
3,a,double,False
4,exchange_code,string,False
5,underlying_code,string,False
6,contract_code,string,False
7,main_contract_code,string,False
8,is_main,bool,False
9,trading_date,date32[day],False
